# Step 1: lock in the RAMP baseline (official code)

**Once:**
1. Kaggle *Add-ons → Secrets*: `HF_TOKEN` = Hugging Face **write** token, attached to this notebook.
2. On your tablet or phone, install the **ntfy** app and subscribe to the topic in `NTFY_TOPIC` below. The first stage installs every dependency and checks all imports, so a missing package stops the run within minutes. Every stage sends you a push when it starts and when it ends: **OK**, **FAILED** (with the error), or **PAUSED** (time budget reached, so run the notebook again).

**Each session:** GPU **T4 x2**, Internet **on**, then **Save Version → Save & Run All**. The notebook stops at the first failed stage, and every stage's full log is uploaded to Hugging Face under `logs/`.

In [1]:
import os
os.environ['HF_REPO'] = 'matokebryan/aat-checkpoints'
os.environ['NTFY_TOPIC'] = 'aat-matoke-7q4r9x'   # subscribe to this in the ntfy app
os.environ['TIME_BUDGET_H'] = '10.5'             # training stops cleanly after this and pushes its checkpoint
MODE = 'scratch'                                 # 'scratch' = thesis baseline (multi-session), 'finetune' = cheap cross-check
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
%cd /kaggle/working
!git clone -q -b matt/funny-planck-3j7md5 https://github.com/mattobryan/AAT.git 2>/dev/null || (cd AAT && git pull -q)
%cd /kaggle/working/AAT

def stage(name, cmd):
    """Run a stage, stream its output, push a phone notification, and stop the notebook if it fails."""
    get_ipython().system(f'python scripts/stage.py {name} "{cmd}"')
    if _exit_code != 0:
        raise SystemExit(f'stage {name} failed (exit {_exit_code}): see the notification or logs/{name}.txt on Hugging Face')

stage('install', 'bash scripts/ramp_official.sh install')   # dependencies + full import check, before anything else
stage('data', 'bash scripts/ramp_official.sh data')         # CIFAR-10 (cached on Hugging Face after the first run)

/kaggle/working
/kaggle/working/AAT
=== stage install started 2026-10-01 20:09 UTC ===
/kaggle/working/AAT/baselines/aat_on_ramp.patch:89: trailing whitespace.
                                eps=eps_s, n_iter=args.l_iters[cur_norm_source], is_train=True)               
/usr/local/lib/python3.12/dist-packages/robustbench/loaders.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
OK    import torch            2.10.0+cu128
OK    import torchvision      0.25.0+cu128
OK    import numpy            2.0.2
OK    import yaml             6.0.3
OK    import huggingface_hub  1.11.0
OK    import autoattack       ok
OK    import robustbench      ok
OK    import geotorch         0.4.0
OK    import timm             1.0.26
OK    official RAMP code      RAMP.py, eat_train.py, MAX.py, eva

## Sanity check: the official pretrained ℓ∞ model must give ≈ 83.7 / 48.1 / 59.8 / 7.7 / union ≈ 7

In [2]:
if not os.path.exists('runs_official/pretr_linf/eval_autoattack.json'):
    stage('sanity', 'bash scripts/ramp_official.sh pretr 0 && python scripts/compare_targets.py --runs runs_official --map pretr_linf=pretr_linf')

=== stage sanity started 2026-10-01 20:10 UTC ===
[hub] nothing to pull for ramp_official/pretr_linf/eval_autoattack.json (RemoteEntryNotFoundError)
sanity already evaluated: /kaggle/working/AAT/runs_official/pretr_linf/eval_autoattack.json
| run | n | metric | ours | paper | Δ | verdict |
|---|---|---|---|---|---|---|
| pretr_linf | 1 | clean | 83.7 | 83.7 | +0.0 | OK |
| pretr_linf | 1 | linf | 48.1 | 48.1 | +0.0 | OK |
| pretr_linf | 1 | l2 | 59.8 | 59.8 | -0.0 | OK |
| pretr_linf | 1 | l1 | 6.5 | 7.7 | -1.2 | CLOSE |

overall: CLOSE
=== stage sanity OK after 0 min (exit 0) ===
No files have been modified since last commit. Skipping to prevent empty commit.
[hub] pushed logs/sanity.txt -> matokebryan/aat-checkpoints
stages.json: 10.1kB [00:00, 26.7MB/s]
[hub] pulled logs/stages.json <- matokebryan/aat-checkpoints
[hub] pushed logs/stages.json -> matokebryan/aat-checkpoints


## A. Thesis baseline: RAMP from scratch (λ=5, 80 epochs, GP), seeds 0 and 1 in parallel
Each session trains until the time budget, pushes its checkpoint, and sends **PAUSED**. Run the notebook again, and it resumes. When both seeds finish, the evaluation runs automatically and sends the comparison with the thesis Table 7.1.

In [3]:
def finished(run):
    return os.path.exists(f'external/ramp/trained_models/{run}/log_eval_final.txt')

if MODE == 'scratch':
    stage('ramp_scratch_train', 'bash scripts/ramp_official.sh train2 ramp_scratch 0 1')
    if all(finished(f'ramp_scratch_l5_s{s}') for s in (0, 1)):
        stage('ramp_scratch_eval', 'bash scripts/ramp_official.sh eval2 ramp_scratch 0 1 && '
              'python scripts/compare_targets.py --runs runs_official --table thesis_table7_1 --map ramp_scratch_official=ramp; '
              'python scripts/compare_targets.py --runs runs_official --table scratch_table3 --map ramp_scratch_official=ramp_l5')
    else:
        print('Training paused at the time budget. Run the notebook again to resume.')

=== stage ramp_scratch_train started 2026-10-01 20:10 UTC ===
done already: ramp_scratch_l5_s1
done already: ramp_scratch_l5_s0
=== stage ramp_scratch_train OK after 0 min (exit 0) ===
Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ...gs/ramp_scratch_train.txt: 100%|██████████████|  66.0B /  66.0B            

Processing Files (1 / 1)      : 100%|██████████████|  66.0B /  66.0B,   ???B/s  
New Data Upload               : 100%|██████████████|  66.0B /  66.0B,   ???B/s  

  ...gs/ramp_scratch_train.txt: 100%|██████████████|  66.0B /  66.0B            

Processing Files (1 / 1)      : 100%|██████████████|  66.0B /  66.0B,  0.00B/s  
New Data Upload               : 100%|██████████████|  66.0B /  66.0B,  0.00B/s  
  ...gs/ramp_scratch_train.txt: 100%|██████████████|  66.0B /  66.0B            
[hub] pushed logs/ramp_scratch_train.txt -> matokebryan/aat-checkpoints
stages.jso

## B. Cross-check: RAMP fine-tuning (paper Table 24), 5 seeds, about 1 GPU-hour each

In [4]:
if MODE == 'finetune':
    stage('ramp_ft_train', 'bash scripts/ramp_official.sh train2 ramp \\"0 2 4\\" \\"1 3\\"')
    stage('ramp_ft_eval', 'bash scripts/ramp_official.sh eval2 ramp \\"0 2 4\\" \\"1 3\\" && '
          'python scripts/compare_targets.py --runs runs_official --map ramp_official=ramp_l1.5 pretr_linf=pretr_linf')